<a href="https://colab.research.google.com/github/userNOTfound-bot/Ollama-Google-Colab/blob/main/Ollama_Google_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚠️ You must have ***Ollama*** installed in your system.
If you didn't had that, then follow these steps to install that.

Installation step:
1. Go to Ollama official website or use this redirect link https://ollama.com
2. Click on the download button and choose your OS, then download & install ollama in your PC/Laptop.

***Installing Ollama in your system is done.***

Now follow these steps in below 👇

In [ ]:
# @title Install Ollama + Cloudflared, then start tunnel

# Step 1: Install dependencies (zstd for extraction, pciutils for GPU detection) and Ollama
!apt-get update && apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

# Step 2: Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb && dpkg -i cloudflared-linux-amd64.deb
!cloudflared -v

# Step 3: Configure environment for GPU support
import os
import asyncio
import subprocess

os.environ['OLLAMA_HOST'] = '0.0.0.0'
# Systemd is missing in Colab, so we manually point to GPU drivers and force GPU detection
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/nvidia/lib:/usr/local/nvidia/lib64'
os.environ['OLLAMA_GPU_OVERRIDE'] = '1'

# Step 4: Run Ollama + Cloudflared together
async def run_command(cmd):
    print(f'>>> starting {" ".join(cmd)}')
    try:
        process = await asyncio.create_subprocess_exec(
            *cmd,
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE
        )

        async def log_stream(stream):
            while True:
                line = await stream.readline()
                if line:
                    decoded_line = line.decode().strip()
                    print(decoded_line)
                    # Check if GPU is detected in logs for confirmation
                    if "inference compute" in decoded_line and "CUDA" in decoded_line:
                        print("\n✅ SUCCESS: GPU Detected and Ready!")
                else:
                    break

        await asyncio.gather(
            log_stream(process.stdout),
            log_stream(process.stderr)
        )
    except FileNotFoundError:
        print(f'Error: The executable "{cmd[0]}" was not found in the path.')
    except asyncio.CancelledError:
        print(f'>>> process {" ".join(cmd)} stopped.')

# Execute services
print("Starting services... Use the TryCloudflare link appearing below to connect.")
await asyncio.gather(
    run_command(['ollama', 'serve']),
    run_command(['cloudflared', 'tunnel', '--url', 'http://localhost:11434']),
)